# Dataset exploration
Audit the actual local extraction before training. Test images are inspected for integrity and overlap only, never used to select model settings. Patient identifiers are inferred from filenames and cannot establish verified patient separation.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

## Inventory and duplicate extraction
Decode each source image, record exact-byte and decoded-pixel hashes, and compare nested copies. Errors remain visible in the audit.

In [ ]:
from ml.data.dataset import audit_dataset
inventory = audit_dataset(config["dataset_root"])
inventory.groupby(["source_split", "label"]).size().unstack()

In [ ]:
import json
json.loads(Path("artifacts/data/audit.json").read_text())

## Distribution and image dimensions
Class imbalance and variable aspect ratios affect training and preprocessing choices.

In [ ]:
import matplotlib.pyplot as plt
inventory.groupby(["source_split", "label"]).size().unstack().plot.bar(title="Original class counts")
plt.show()
inventory[["width", "height"]].describe()

In [ ]:
inventory.plot.scatter(x="width", y="height", alpha=0.15, title="Image dimensions")
plt.show()
inventory.groupby("format").size()

## Representative training images
Only development images are visualized here. Outputs containing images stay local and are cleared before commit.

In [ ]:
from PIL import Image
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, label in zip(axes, ["NORMAL", "PNEUMONIA"]):
    row = inventory[(inventory.source_split == "train") & (inventory.label == label)].iloc[0]
    with Image.open(Path(config["dataset_root"]) / row.path) as image:
        ax.imshow(image, cmap="gray")
    ax.set_title(label)
    ax.axis("off")
plt.show()

## Leakage and limitations
Review `artifacts/data/duplicates.csv` and `audit.json`. Filename-derived groups and pixel hashes detect some leakage, not all repeated patients or near duplicates. The original validation set is too small for reliable selection. Notebook 02 derives validation only from the training pool; the original test set remains fixed.